# ScreamingFace · service connections

Connect the model providers and tool services advertised by your configured ScreamingFace engine,
then see how execution preflight prevents repeated authentication failures before any model spend.

Start the local stack first:

```bash
cd packages/screamingface/apps/screamingface-engine
./dev.sh
```

For model providers the boundary is **SDK → screamingface-engine → AI Gateway → provider**.
Tavily is engine-owned: **SDK → screamingface-engine → Tavily**. The SDK never sends credentials
directly to AI Gateway, a model provider, or Tavily.

## 1 · Open the provider panel

In [ ]:
import screamingface as sf

panel = sf.connect()
panel

The panel reads fresh status from the configured engine and shows where credentials
will be stored. API-key inputs are masked and cleared after every attempt. OAuth displays an
authorization link only after you press its button; it **does not open a browser automatically**.

For model providers, connected means the engine's AI Gateway holds a credential. For Tavily, it
means Tavily validated the key and this local engine process currently holds it; restart requires
reconnection. Neither status claims that every advertised model or future tool action will
succeed.

## 2 · Read status from Python

In [ ]:
connections = sf.connections.list()
connections

Scripts use explicit targeted calls and never receive a hidden terminal prompt.
OAuth returns a bounded `OAuthFlow`; API keys travel once in the private request body. These
examples are comments so running this guide never starts or replaces a connection unexpectedly.

In [ ]:
# OAuth — inspect flow.authorize_url, then call flow.wait() after authorizing:
# flow = sf.connect("codex", method="oauth")

# API key — read it from your process environment, never a shared notebook literal:
# import os
# gemini = sf.connect("gemini", api_key=os.environ["GEMINI_API_KEY"])

# Tavily is validated directly and retained only by this local engine process:
# tavily = sf.connect("tavily", api_key=os.environ["TAVILY_API_KEY"])
# sf.connections.get("tavily")

# Disconnect is idempotent:
# sf.disconnect("gemini")

## 3 · Execution checks requirements once

In [ ]:
def connection_actions(error: sf.ConnectionRequiredError) -> dict[str, object]:
    """Turn one preflight error into script-friendly details."""

    return {
        "providers": error.providers,
        "models": error.models,
        "roles": error.roles,
        "message": str(error),
    }

`fusion.run(...)` checks member and model-reducer providers. `run.grade()` checks a
model judge only when the benchmark uses one. `fusion.evaluate(...)` checks their union once before
the first request. Missing credentials raise one actionable `ConnectionRequiredError`, not one
failure per case. This guide performs **no paid model call**.

Dataset access remains separate. GPQA and other Hugging Face datasets use the researcher's native
Hugging Face session; `sf.connect()` never receives `HF_TOKEN` or dataset credentials. This phase
connects Tavily but does not yet route model tools through it.